# Initial Libraies

In [1]:
!pip install dwave-ocean-sdk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.1/119.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00


In [2]:
!pip install dwave-neal

In [3]:
!pip install job-shop-lib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.1/293.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from job_shop_lib.visualization.gantt import plot_gantt_chart
from job_shop_lib import JobShopInstance, Operation
from job_shop_lib.dispatching import (
    ReadyOperationsFilterType,
    create_composite_operation_filter,
)
from job_shop_lib.dispatching.rules import (
    DispatchingRuleSolver,
    DispatchingRuleType,
)

from job_shop_lib.benchmarking import load_benchmark_instance

from collections import defaultdict, deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List
from tqdm import tqdm

from collections import Counter

from dwave.system.samplers import DWaveSampler
from dwave.system.composites import EmbeddingComposite
import dwave.inspector as inspector
from dimod import ConstrainedQuadraticModel, CQM, SampleSet
from dwave.system import LeapHybridCQMSampler
from dimod.vartypes import Vartype
from dimod import Binary, quicksum
from dimod import BinaryQuadraticModel
import dimod

from neal import SimulatedAnnealingSampler

In [5]:
plt.style.use("ggplot")

In [6]:
def plot_gantt_chart_from_dispatching_rule(
    dispatching_rule: DispatchingRuleType,
    instance: JobShopInstance,
    optimization_level: int = 0,
) -> plt.Figure:

    if optimization_level == 0:
        pruning_function = None
    elif optimization_level == 1:
        pruning_function = ReadyOperationsFilterType.DOMINATED_OPERATIONS
    elif optimization_level == 2:
        pruning_function = create_composite_operation_filter(
            [
                ReadyOperationsFilterType.DOMINATED_OPERATIONS,
                ReadyOperationsFilterType.NON_IMMEDIATE_MACHINES,
            ]
        )
    else:
        raise ValueError(f"Invalid optimization level: {optimization_level}")

    solver = DispatchingRuleSolver(
        dispatching_rule,
        ready_operations_filter=pruning_function,
    )
    solution = solver.solve(instance)

    title = f"{instance.name} - {dispatching_rule.name} (optimization level {optimization_level})"
    number_of_x_ticks = 10 if solution.makespan() >= 1000 else 15
    fig, ax = plot_gantt_chart(
        solution, title=title, number_of_x_ticks=number_of_x_ticks
    )
    if instance.num_jobs > 20:
        # Remove legend if there are too many jobs
        ax.legend().remove()
    return fig

In [7]:
class JSPMUInstance:

    def __init__(self, name=""):
        self.name = name
        self.n_jobs = 0
        self.n_machines = 0

        # jobs[j] = [(machine, duration), ...]
        self.jobs = []

        # machine_holes[m] = [(start, end), ...]
        self.machine_holes = {}

    @staticmethod
    def read(filepath):

        instance = JSPMUInstance(Path(filepath).stem)

        with open(filepath, "r") as f:
            lines = [l.strip() for l in f if l.strip()]

        # ---- header ----
        header = lines[0].split()
        instance.n_jobs = int(header[0])
        instance.n_machines = int(header[1])

        # ---- jobs ----
        for j in range(instance.n_jobs):

            tokens = list(map(int, lines[1 + j].split()))

            if len(tokens) != 2 * instance.n_machines:
                raise ValueError(
                    f"Job {j} has wrong number of entries."
                )

            ops = []

            for k in range(0, len(tokens), 2):

                machine = tokens[k]
                duration = tokens[k + 1]

                if machine < 0 or machine >= instance.n_machines:
                    raise ValueError(f"Invalid machine {machine}")

                ops.append((machine, duration))

            instance.jobs.append(ops)

        # ---- initialize empty holes ----
        for m in range(instance.n_machines):
            instance.machine_holes[m] = []

        # ---- holes section ----
        idx = 1 + instance.n_jobs

        if idx < len(lines):

            if lines[idx] != "[MACHINE_HOLES]":
                raise ValueError("Missing [MACHINE_HOLES] section")

            idx += 1

            while idx < len(lines):

                tokens = list(map(int, lines[idx].split()))

                machine = tokens[0]
                n_holes = tokens[1]

                expected = 2 + 2 * n_holes
                if len(tokens) != expected:
                    raise ValueError(
                        f"Wrong hole format for machine {machine}"
                    )

                holes = []

                pos = 2
                for _ in range(n_holes):

                    start = tokens[pos]
                    duration = tokens[pos + 1]

                    if duration <= 0:
                        raise ValueError("Hole duration must be positive")

                    holes.append((start, start + duration))

                    pos += 2

                instance.machine_holes[machine] = holes

                idx += 1

        return instance

    def __str__(self) -> str:
        print('Name:',self.name)
        print("n jobs:",self.n_jobs)
        print("n machines:",self.n_machines)
        print("jobs:")
        for j in self.jobs:
          print(j)
        print("Unavailabilities:")
        for h in self.machine_holes:
          print('Machine',h,':',self.machine_holes[h])
        return ''

    def get_naive_t(self):
      naive_t = 0
      for j in self.jobs:
        durs = [op[1] for op in j]
        naive_t += sum(durs)
      return naive_t

In [8]:
def to_job_shop_lib_instance(jspmu_instance, name=None, accept_unavailabilities = False):
    """
    Convert a JSPMUInstance into a job_shop_lib.JobShopInstance.

    Conventions:
    - Original jobs keep their original routing.
    - Each machine unavailability (start, end) becomes a dummy one-operation job.
    - metadata["release_dates"]:
        * 0 for original jobs
        * hole start for dummy maintenance jobs
    - metadata["deadlines"]:
        * float("inf") for original jobs
        * hole end for dummy maintenance jobs
    """

    jobs = []
    release_dates = []
    deadlines = []

    # --- Original jobs ---
    # jspmu_instance.jobs[j] = [(machine, duration), ...]
    for job in jspmu_instance.jobs:
        converted_job = [Operation(machine, duration) for machine, duration in job]
        jobs.append(converted_job)
        release_dates.append(0)
        deadlines.append(float("inf"))

    if accept_unavailabilities:
      # --- Dummy jobs for machine holes ---
      # jspmu_instance.machine_holes[m] = [(start, end), ...]
      # Each hole becomes a 1-operation job of duration end-start on machine m
      for machine in sorted(jspmu_instance.machine_holes):
          holes = jspmu_instance.machine_holes[machine]
          for start, end in holes:
              duration = end - start
              if duration <= 0:
                  raise ValueError(
                      f"Invalid hole on machine {machine}: ({start}, {end})"
                  )

              dummy_job = [Operation(machine, duration)]
              jobs.append(dummy_job)
              release_dates.append(start)
              deadlines.append(end)

    instance = JobShopInstance(
        jobs=jobs,
        name=name if name is not None else jspmu_instance.name
    )

    instance.metadata["release_dates"] = release_dates
    instance.metadata["deadlines"] = deadlines

    return instance

# Import instances

In [9]:
# inst = JSPMUInstance.read("abz5_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu")

In [10]:
# print(inst)

In [11]:
# inst.get_naive_t()

# Create BQM

In [12]:
def build_jspmu_bqm(
    inst,
    horizon,
    lambda_1=10.0,
    lambda_2=10.0,
    lambda_3=10.0,
):
    """
    Build a BINARY BQM for a Job Shop Scheduling Problem with machine unavailabilities,
    following the paper's approach:
      - real operations are modeled with time-indexed binary variables x[j,i,t]
      - fixed machine unavailabilities are treated as fixed dummy one-operation jobs
        and are incorporated through the machine-conflict penalty P2

    Parameters
    ----------
    inst : object
        Instance object with attributes:
          - inst.jobs : list of jobs; each job is a list of (machine, proc_time)
          - inst.machine_holes : dict-like or list-like structure such that
                                 inst.machine_holes[m] = [(start, end), ...]
          - inst.n_jobs
          - inst.n_machines
    horizon : int
        Time horizon H. Start times are allowed in {0, ..., H-p}.
    lambda_1 : float
        Penalty weight for "start exactly once" (P1).
    lambda_2 : float
        Penalty weight for machine conflicts (P2), including conflicts with
        fixed machine unavailability intervals.
    lambda_3 : float
        Penalty weight for within-job precedence (P3).

    Returns
    -------
    bqm : dimod.BinaryQuadraticModel
        BQM in vartype BINARY.
    metadata : dict
        Helpful metadata about variables and operations.
    """

    if horizon <= 0:
        raise ValueError("horizon must be a positive integer")

    jobs = inst.jobs
    n_jobs = len(jobs)

    # --- helper functions -----------------------------------------------------

    def var_label(j, i, t):
        # j = job index, i = operation index within job, t = start time
        return (j, i, t)

    def intervals_overlap(s1, p1, s2, p2):
        """Return True if [s1, s1+p1) overlaps [s2, s2+p2)."""
        return (s1 < s2 + p2) and (s2 < s1 + p1)

    def get_holes(machine):
        # works whether machine_holes is a dict or a list
        holes = inst.machine_holes[machine]
        return holes if holes is not None else []

    # --- collect operation data -----------------------------------------------

    operations = []  # each element: dict with job, op, machine, p
    machine_to_ops = defaultdict(list)

    for j, job in enumerate(jobs):
        for i, (machine, p) in enumerate(job):
            if p <= 0:
                raise ValueError(f"Processing time must be positive, got p={p} for job {j}, op {i}")
            if p > horizon:
                raise ValueError(
                    f"Operation (job={j}, op={i}) has p={p} > horizon={horizon}; "
                    "increase horizon."
                )

            op = {
                "job": j,
                "op": i,
                "machine": machine,
                "p": p,
                "valid_starts": list(range(0, horizon - p + 1)),
            }
            operations.append(op)
            machine_to_ops[machine].append(op)

    # --- create BQM -----------------------------------------------------------

    bqm = dimod.BinaryQuadraticModel(vartype=dimod.BINARY)

    # Add all variables explicitly with 0 linear bias
    for op in operations:
        j, i = op["job"], op["op"]
        for t in op["valid_starts"]:
            bqm.add_variable(var_label(j, i, t), 0.0)

    # -------------------------------------------------------------------------
    # Objective: minimize sum of start times of the last operation of each job
    # f(x) = sum_j sum_t t * x[last_op_of_job_j, t]
    # -------------------------------------------------------------------------
    for j, job in enumerate(jobs):
        last_i = len(job) - 1
        p_last = job[last_i][1]
        for t in range(0, horizon - p_last + 1):
            bqm.add_linear(var_label(j, last_i, t), float(t))

    # -------------------------------------------------------------------------
    # P1: each operation starts exactly once
    #   (sum_t x_t - 1)^2
    # For binary x:
    #   = 1 - sum_t x_t + 2 * sum_{t<t'} x_t x_t'
    # -------------------------------------------------------------------------
    for op in operations:
        j, i = op["job"], op["op"]
        starts = op["valid_starts"]

        bqm.offset += lambda_1

        for t in starts:
            bqm.add_linear(var_label(j, i, t), -lambda_1)

        for idx1 in range(len(starts)):
            t1 = starts[idx1]
            v1 = var_label(j, i, t1)
            for idx2 in range(idx1 + 1, len(starts)):
                t2 = starts[idx2]
                v2 = var_label(j, i, t2)
                bqm.add_quadratic(v1, v2, 2.0 * lambda_1)

    # -------------------------------------------------------------------------
    # P2: machine conflicts between real operations
    # Add lambda_2 * x[j,i,t] * x[j',i',t'] whenever two operations on the
    # same machine overlap in time.
    # -------------------------------------------------------------------------
    for machine, ops_on_machine in machine_to_ops.items():
        n_ops_m = len(ops_on_machine)
        for a in range(n_ops_m):
            op_a = ops_on_machine[a]
            j1, i1, p1 = op_a["job"], op_a["op"], op_a["p"]

            for b in range(a + 1, n_ops_m):
                op_b = ops_on_machine[b]
                j2, i2, p2 = op_b["job"], op_b["op"], op_b["p"]

                for t1 in op_a["valid_starts"]:
                    v1 = var_label(j1, i1, t1)
                    for t2 in op_b["valid_starts"]:
                        if intervals_overlap(t1, p1, t2, p2):
                            v2 = var_label(j2, i2, t2)
                            bqm.add_quadratic(v1, v2, lambda_2)

    # -------------------------------------------------------------------------
    # P2 extension for fixed machine unavailabilities:
    # each hole [a,b) is treated like a fixed dummy one-operation job with
    # start fixed at a and duration (b-a). Since its binary variable is fixed
    # to 1, the quadratic conflict term reduces to a linear penalty on any
    # real operation start that overlaps the hole.
    # -------------------------------------------------------------------------
    for machine, ops_on_machine in machine_to_ops.items():
        holes = get_holes(machine)

        for (a, b) in holes:
            hole_start = int(a)
            hole_end = int(b)
            hole_len = hole_end - hole_start

            if hole_len <= 0:
                raise ValueError(f"Invalid hole on machine {machine}: {(a, b)}")

            for op in ops_on_machine:
                j, i, p = op["job"], op["op"], op["p"]

                for t in op["valid_starts"]:
                    if intervals_overlap(t, p, hole_start, hole_len):
                        # conflict with fixed dummy operation => linear term
                        bqm.add_linear(var_label(j, i, t), lambda_2)

    # -------------------------------------------------------------------------
    # P3: precedence inside each job
    # Add lambda_3 * x[j,i,t] * x[j,i+1,t'] whenever t + p_i > t'
    # -------------------------------------------------------------------------
    for j, job in enumerate(jobs):
        for i in range(len(job) - 1):
            p_i = job[i][1]
            p_next = job[i + 1][1]

            starts_i = range(0, horizon - p_i + 1)
            starts_next = range(0, horizon - p_next + 1)

            for t in starts_i:
                v1 = var_label(j, i, t)
                cutoff = t + p_i  # next op cannot start before this

                for t_next in starts_next:
                    if t_next < cutoff:
                        v2 = var_label(j, i + 1, t_next)
                        bqm.add_quadratic(v1, v2, lambda_3)

    metadata = {
        "name": getattr(inst, "name", None),
        "n_jobs": n_jobs,
        "n_machines": getattr(inst, "n_machines", None),
        "num_real_operations": len(operations),
        "num_variables": bqm.num_variables,
        "num_interactions": bqm.num_interactions,
        "horizon": horizon,
        "lambdas": {
            "lambda_1": lambda_1,
            "lambda_2": lambda_2,
            "lambda_3": lambda_3,
        },
        "variable_format": "(job_index, op_index, start_time)",
    }

    return bqm, metadata

In [13]:
# bqm, meta = build_jspmu_bqm(
#     inst,
#     horizon=inst.get_naive_t(),
#     lambda_1=20.0,
#     lambda_2=20.0,
#     lambda_3=20.0,
# )

In [14]:
# print(meta)
# print("offset:", bqm.offset)
# print("vars:", bqm.num_variables)
# print("quadratic terms:", bqm.num_interactions)

In [15]:
# bqm, meta = build_jspmu_bqm(
#     inst,
#     horizon = 39,
#     lambda_1=20.0,
#     lambda_2=20.0,
#     lambda_3=20.0,
# )

In [16]:
# print(meta)
# print("offset:", bqm.offset)
# print("vars:", bqm.num_variables)
# print("quadratic terms:", bqm.num_interactions)

In [17]:
# meta['num_variables']

# Variable and instance QUBO size analysis

In [18]:
import zipfile
import os

In [19]:
df = pd.read_csv('heuristic_H_Non_Preemptive_results_rescaled.csv')
df.head()

,id,H-Preemptive
0,abz5,14
1,abz6,11
2,ft06,-1
3,ft10,14
4,la01,11


In [20]:
df.index = df['id']

In [21]:
makespans_H = df.rename(columns = {'H-Preemptive':'T'}).drop(columns = ['id']).T.to_dict()

In [22]:
makespans_H

{'abz5': {'T': 14},
 'abz6': {'T': 11},
 'ft06': {'T': -1},
 'ft10': {'T': 14},
 'la01': {'T': 11},
 'la02': {'T': 9},
 'la03': {'T': 11},
 'la04': {'T': 13},
 'la05': {'T': 12},
 'la16': {'T': 13},
 'la17': {'T': 10},
 'la18': {'T': 14},
 'la19': {'T': 15},
 'la20': {'T': 15},
 'orb01': {'T': 11},
 'orb02': {'T': 14},
 'orb03': {'T': 12},
 'orb04': {'T': 14},
 'orb05': {'T': 12},
 'orb06': {'T': 16},
 'orb07': {'T': 13},
 'orb08': {'T': 14},
 'orb09': {'T': 13},
 'orb10': {'T': 10}}

In [23]:
zip_path = "generated_jspmu_instances_literature_rescaled_3.zip"
extract_dir = "generated_jspmu_instances_literature_rescaled"

# List to store file names inside the zip
file_names = []

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    file_names = zip_ref.namelist()  # names of all files/folders inside the zip
    zip_ref.extractall(extract_dir)  # unzip everything

print("Files inside zip:")
print(file_names)

Files inside zip:
['abz5_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'abz6_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'ft06_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'ft10_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la01_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la02_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la03_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la04_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la05_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la16_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la17_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la18_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la19_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'la20_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'orb01_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'orb02_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'orb03_mu_mtbf300_durlognorm35_03_op1-3_hole1-1.jspmu', 'orb04_mu_mtbf300_durlognorm35_03_op1-3_hol

In [24]:
data = []
problematics = []

for instance_name in tqdm(file_names):
  try:
    jspmu_instance = JSPMUInstance.read(f'{extract_dir}/{instance_name}')
    machine_unavailability = jspmu_instance.machine_holes
    id_instance = instance_name.split('_')[0]
    row = {'id':id_instance}

    t_naive = jspmu_instance.get_naive_t()
    row['naive_t'] = t_naive
    bqm, meta = build_jspmu_bqm(
    jspmu_instance,
    horizon = t_naive,
    lambda_1=20.0,
    lambda_2=20.0,
    lambda_3=20.0,
    )
    n_variables = meta['num_variables']
    n_interactions = meta['num_interactions']

    row['n_var_naive'] = n_variables
    row['n_interactions_naive'] = n_interactions

    H_T = makespans_H[id_instance]['T']
    row['H-T'] = H_T

    bqm1, meta1 = build_jspmu_bqm(
    jspmu_instance,
    horizon = H_T,
    lambda_1=20.0,
    lambda_2=20.0,
    lambda_3=20.0,
    )

    n_variables_H = meta1['num_variables']
    n_interactions_H = meta1['num_interactions']

    row['n_var_H'] = n_variables_H
    row['n_interactions_H'] = n_interactions_H

    data.append(row)

  except:
    problematics.append(instance_name)

100%|██████████| 24/24 [00:00<00:00, 89.30it/s]


In [25]:
pd.DataFrame(data)

,id,naive_t,n_var_naive,n_interactions_naive,H-T,n_var_H,n_interactions_H
0,abz5,20,169,3475,14,115,1774
1,abz6,16,137,2178,11,92,1068
2,ft10,17,145,2352,14,118,1614
3,la01,16,137,2177,11,92,1067
4,la02,13,113,1459,9,77,729
5,la03,16,137,2107,11,92,1022
6,la04,17,145,2449,13,109,1467
7,la05,18,153,2760,12,99,1275
8,la16,16,137,2243,13,110,1517
9,la17,16,137,2170,10,83,895


In [26]:
pd.DataFrame(data).to_csv('n_variables_naiveT_vs_H-T.csv',index = False)